In [1]:
import os
import glob
import numpy as np
import pandas as pd
import tifffile as tiff
import matplotlib.pyplot as plt

from scipy.ndimage import gaussian_filter, median_filter, center_of_mass
from scipy.spatial.distance import cdist
from skimage.registration import phase_cross_correlation
from skimage.feature import peak_local_max, match_template
from scipy.ndimage import shift as ndi_shift

C:\ProgramData\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


In [87]:
CONFIG = {
    "INPUT_PATH": r"C:\Users\luoyi\Desktop\Data\Method\ED_tracking\meas17.tif",
    "OUTPUT_DIR": r"C:\Users\luoyi\Desktop\Data\Method\ED_tracking",

    "REFERENCE_FRAME_INDEX": 0,
    "MANUAL_CENTER_YX": np.array([228, 288], dtype=np.float64),

    "DIRECT_BEAM_TEMPLATE_HALF_SIZE": 6,
    "DIRECT_BEAM_SEARCH_RADIUS": 15,
    "DIRECT_BEAM_USE_LOG": False,

    "MEDIAN_SIZE": 2,
    "GAUSSIAN_SIGMA": 1.0,
    "BACKGROUND_SIGMA": 2.0,
    "USE_LOG_SCALE_FOR_DETECTION": True,

    "NUM_SPOTS": 30,
    "MIN_DISTANCE_BETWEEN_SPOTS": 12,
    "PEAK_THRESHOLD_REL": 0.15,
    "EXCLUDE_CENTER_RADIUS": 10,
    "EXCLUDE_EDGE_MARGIN": 5,
    "TEMPLATE_HALF_SIZE": 7,
    "SEARCH_RADIUS": 8,

    "SUBPIXEL_NEIGHBORHOOD": 1,
}

In [63]:
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def load_tiff_data(path):
    if os.path.isdir(path):
        files = sorted(
            glob.glob(os.path.join(path, "*.tif")) +
            glob.glob(os.path.join(path, "*.tiff"))
        )
        if len(files) == 0:
            raise FileNotFoundError("No TIFF files found in folder.")
        stack = [tiff.imread(f) for f in files]
        return np.array(stack)

    elif os.path.isfile(path):
        arr = tiff.imread(path)
        if arr.ndim == 2:
            arr = arr[np.newaxis, ...]
        elif arr.ndim != 3:
            raise ValueError(f"Unsupported TIFF shape: {arr.shape}")
        return arr

    else:
        raise FileNotFoundError(f"Input path not found: {path}")


def preprocess_frame(frame,
                     median_size=3,
                     gaussian_sigma=1.0,
                     background_sigma=12.0,
                     use_log=False):
    f = frame.astype(np.float32)

    if median_size > 1:
        f = median_filter(f, size=median_size)

    if gaussian_sigma > 0:
        f = gaussian_filter(f, sigma=gaussian_sigma)

    if background_sigma > 0:
        bg = gaussian_filter(f, sigma=background_sigma)
        f = f - bg
        f[f < 0] = 0

    if use_log:
        f = np.log1p(f)

    return f


def preprocess_stack(stack, median_size=3, gaussian_sigma=1.0,
                     background_sigma=12.0, use_log=False):
    out = np.zeros_like(stack, dtype=np.float32)
    for i in range(stack.shape[0]):
        out[i] = preprocess_frame(
            stack[i],
            median_size=median_size,
            gaussian_sigma=gaussian_sigma,
            background_sigma=background_sigma,
            use_log=use_log
        )
    return out


def extract_patch(image, center_yx, half_size):
    h, w = image.shape
    cy, cx = center_yx
    cy_i, cx_i = int(round(cy)), int(round(cx))

    y0 = cy_i - half_size
    y1 = cy_i + half_size + 1
    x0 = cx_i - half_size
    x1 = cx_i + half_size + 1

    if y0 < 0 or x0 < 0 or y1 > h or x1 > w:
        return None, (None, None, None, None)

    return image[y0:y1, x0:x1], (y0, y1, x0, x1)


def subpixel_peak_from_correlation(corr_map, peak_yx, neighborhood=1):
    py, px = peak_yx
    h, w = corr_map.shape

    y0 = max(py - neighborhood, 0)
    y1 = min(py + neighborhood + 1, h)
    x0 = max(px - neighborhood, 0)
    x1 = min(px + neighborhood + 1, w)

    local = corr_map[y0:y1, x0:x1].copy()
    local = local - local.min()

    if local.sum() <= 0:
        return float(py), float(px)

    cy, cx = center_of_mass(local)
    return y0 + cy, x0 + cx


def track_one_spot_template(search_image,
                            template,
                            predicted_yx,
                            search_radius=8,
                            subpixel_neighborhood=1):
    th, tw = template.shape
    half_th = th // 2
    half_tw = tw // 2

    h, w = search_image.shape
    py, px = predicted_yx

    y0 = int(np.floor(py)) - search_radius - half_th
    y1 = int(np.floor(py)) + search_radius + half_th + 1
    x0 = int(np.floor(px)) - search_radius - half_tw
    x1 = int(np.floor(px)) + search_radius + half_tw + 1

    if y0 < 0 or x0 < 0 or y1 > h or x1 > w:
        return None, np.nan

    roi = search_image[y0:y1, x0:x1]
    if roi.shape[0] < th or roi.shape[1] < tw:
        return None, np.nan

    corr = match_template(roi, template, pad_input=False)
    peak_idx = np.unravel_index(np.argmax(corr), corr.shape)
    peak_val = corr[peak_idx]

    sub_y, sub_x = subpixel_peak_from_correlation(
        corr, peak_idx, neighborhood=subpixel_neighborhood
    )

    matched_y = y0 + sub_y + half_th
    matched_x = x0 + sub_x + half_tw

    return np.array([matched_y, matched_x], dtype=np.float64), float(peak_val)


def shift_image(image, shift_yx, order=1):
    return ndi_shift(image, shift=shift_yx, order=order, mode='constant', cval=0.0)

In [65]:
def extract_direct_beam_template(ref_frame, manual_center_yx, half_size):
    template, _ = extract_patch(ref_frame, manual_center_yx, half_size)
    if template is None:
        raise ValueError("Direct beam template is out of image bounds.")
    return template


def track_direct_beam_stack(beam_tracking_stack,
                            manual_center_yx,
                            template_half_size,
                            search_radius,
                            subpixel_neighborhood,
                            ref_frame_index=0):
    ref_frame = beam_tracking_stack[ref_frame_index]
    beam_template = extract_direct_beam_template(
        ref_frame, manual_center_yx, template_half_size
    )

    n_frames = beam_tracking_stack.shape[0]
    beam_positions = np.full((n_frames, 2), np.nan, dtype=np.float64)
    beam_scores = np.full(n_frames, np.nan, dtype=np.float64)

    for i in range(n_frames):
        matched_yx, score = track_one_spot_template(
            beam_tracking_stack[i],
            beam_template,
            predicted_yx=manual_center_yx,
            search_radius=search_radius,
            subpixel_neighborhood=subpixel_neighborhood
        )
        if matched_yx is not None:
            beam_positions[i] = matched_yx
            beam_scores[i] = score

    return beam_positions, beam_scores, beam_template

In [67]:
def center_stack_by_direct_beam(stack, beam_positions):
    n_frames, h, w = stack.shape
    image_center_yx = np.array([h / 2, w / 2], dtype=np.float64)

    centered_stack = np.zeros_like(stack, dtype=np.float32)
    centering_shifts = np.full((n_frames, 2), np.nan, dtype=np.float64)

    for i in range(n_frames):
        if not np.all(np.isfinite(beam_positions[i])):
            continue

        shift_yx = image_center_yx - beam_positions[i]
        centering_shifts[i] = shift_yx
        centered_stack[i] = shift_image(stack[i].astype(np.float32), shift_yx, order=1)

    return centered_stack, centering_shifts, image_center_yx

In [69]:
def exclude_mask(shape, center=None, radius=30, edge_margin=20):
    h, w = shape
    yy, xx = np.indices((h, w))
    mask = np.zeros((h, w), dtype=bool)

    if center is None:
        cy, cx = h / 2, w / 2
    else:
        cy, cx = center

    rr = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    mask |= (rr < radius)

    mask[:edge_margin, :] = True
    mask[-edge_margin:, :] = True
    mask[:, :edge_margin] = True
    mask[:, -edge_margin:] = True
    return mask


def detect_reference_spots(ref_frame_processed,
                           num_spots,
                           min_distance,
                           threshold_rel,
                           exclude_center_radius,
                           exclude_edge_margin,
                           center):
    mask = exclude_mask(
        ref_frame_processed.shape,
        center=center,
        radius=exclude_center_radius,
        edge_margin=exclude_edge_margin
    )

    work = ref_frame_processed.copy()
    work[mask] = 0

    coords = peak_local_max(
        work,
        min_distance=min_distance,
        threshold_rel=threshold_rel,
        num_peaks=num_spots * 3
    )

    if len(coords) == 0:
        raise RuntimeError("No peaks detected. Try lowering threshold_rel.")

    intensities = work[coords[:, 0], coords[:, 1]]
    order = np.argsort(intensities)[::-1]
    coords = coords[order][:num_spots]
    return coords.astype(np.float64)


def build_spot_templates(ref_proc, ref_spots, template_half_size):
    templates = []
    valid_spots = []

    for pt in ref_spots:
        template, _ = extract_patch(ref_proc, pt, template_half_size)
        if template is not None:
            templates.append(template)
            valid_spots.append(pt)

    return np.array(valid_spots, dtype=np.float64), templates


def track_bragg_spots_stack(processed_stack,
                            ref_spots,
                            templates,
                            search_radius,
                            subpixel_neighborhood,
                            ref_frame_index=0):
    n_frames = processed_stack.shape[0]
    num_spots = len(ref_spots)

    tracked = np.full((n_frames, num_spots, 2), np.nan, dtype=np.float64)
    scores = np.full((n_frames, num_spots), np.nan, dtype=np.float64)

    tracked[ref_frame_index] = ref_spots
    scores[ref_frame_index] = 1.0

    for i in range(n_frames):
        frame_proc = processed_stack[i]
        for j in range(num_spots):
            matched_yx, score = track_one_spot_template(
                frame_proc,
                templates[j],
                predicted_yx=ref_spots[j],
                search_radius=search_radius,
                subpixel_neighborhood=subpixel_neighborhood
            )
            if matched_yx is not None:
                tracked[i, j] = matched_yx
                scores[i, j] = score

    return tracked, scores

In [71]:
def compute_frame_centers(tracked):
    n_frames = tracked.shape[0]
    frame_centers = np.full((n_frames, 2), np.nan, dtype=np.float64)

    for i in range(n_frames):
        coords = tracked[i]
        valid = np.isfinite(coords[:, 0])
        if np.sum(valid) > 0:
            frame_centers[i, 0] = np.mean(coords[valid, 0])
            frame_centers[i, 1] = np.mean(coords[valid, 1])

    return frame_centers


def compute_radial_angular(coords_yx, center_yx):
    dy = coords_yx[:, 0] - center_yx[0]
    dx = coords_yx[:, 1] - center_yx[1]
    r = np.sqrt(dx**2 + dy**2)
    theta = np.arctan2(dy, dx)
    return r, theta


def build_tracking_dataframe(tracked,
                             scores,
                             ref_spots,
                             frame_centers,
                             beam_positions,
                             beam_scores,
                             centering_shifts,
                             center_yx):
    n_frames, num_spots, _ = tracked.shape
    rows = []

    for i in range(n_frames):
        for j in range(num_spots):
            y, x = tracked[i, j]
            sy = y - ref_spots[j, 0] if np.isfinite(y) else np.nan
            sx = x - ref_spots[j, 1] if np.isfinite(x) else np.nan
            disp = np.sqrt(sx**2 + sy**2) if np.isfinite(sx) else np.nan

            rows.append({
                "frame": i,
                "spot_id": j,
                "y": y,
                "x": x,
                "dy_from_ref": sy,
                "dx_from_ref": sx,
                "disp_from_ref": disp,
                "corr_score": scores[i, j],
                "ref_y": ref_spots[j, 0],
                "ref_x": ref_spots[j, 1],
                "frame_center_y": frame_centers[i, 0],
                "frame_center_x": frame_centers[i, 1],
                "dy_from_frame_center": y - frame_centers[i, 0] if np.isfinite(y) and np.isfinite(frame_centers[i, 0]) else np.nan,
                "dx_from_frame_center": x - frame_centers[i, 1] if np.isfinite(x) and np.isfinite(frame_centers[i, 1]) else np.nan,
                "direct_beam_y": beam_positions[i, 0],
                "direct_beam_x": beam_positions[i, 1],
                "direct_beam_score": beam_scores[i],
                "centering_shift_dy": centering_shifts[i, 0],
                "centering_shift_dx": centering_shifts[i, 1],
            })

    df = pd.DataFrame(rows)

    for i in range(n_frames):
        coords = tracked[i]
        valid = np.isfinite(coords[:, 0])

        r = np.full(num_spots, np.nan)
        theta = np.full(num_spots, np.nan)

        if np.any(valid):
            rv, thetav = compute_radial_angular(coords[valid], center_yx)
            r[valid] = rv
            theta[valid] = thetav

        df.loc[df["frame"] == i, "r"] = r
        df.loc[df["frame"] == i, "theta_rad"] = theta

    return df

In [89]:
def plot_trajectories(tracked,
                      frame_centers,
                      output_path,
                      spot_ids=None,
                      mode='color',
                      figsize=(6, 6),
                      point_size=2,
                      line_width=1.0):
    import numpy as np
    import matplotlib.pyplot as plt

    n_frames, num_spots, _ = tracked.shape

    if spot_ids is None:
        spot_ids = range(num_spots)

    plt.figure(figsize=figsize)
    scatter_obj = None

    for j in spot_ids:
        xy = tracked[:, j, :]
        valid = np.isfinite(xy[:, 0]) & np.isfinite(frame_centers[:, 0])

        if np.sum(valid) <= 1:
            continue

        rel_x = xy[valid, 1] - frame_centers[valid, 1]
        rel_y = xy[valid, 0] - frame_centers[valid, 0]

        if mode == 'color':
            frames_valid = np.arange(n_frames)[valid]
            plt.plot(rel_x, rel_y, '-', lw=line_width, alpha=0.4)
            scatter_obj = plt.scatter(rel_x, rel_y, c=frames_valid, s=point_size)

        elif mode == 'minimal':
            plt.plot(rel_x, rel_y, '-', lw=line_width, color='black')
            plt.scatter(rel_x, rel_y, s=point_size / 2, color='black')

    plt.axhline(0, linestyle='--', linewidth=0.8)
    plt.axvline(0, linestyle='--', linewidth=0.8)
    plt.xlabel("Δx (pixel)")
    plt.ylabel("Δy (pixel)")

    if mode == 'color' and scatter_obj is not None:
        plt.colorbar(scatter_obj, label="Frame")

    plt.axis('equal')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

In [75]:
def run_pipeline(config):
    ensure_dir(config["OUTPUT_DIR"])

    stack = load_tiff_data(config["INPUT_PATH"])
    n_frames, h, w = stack.shape

    beam_tracking_stack = preprocess_stack(
        stack,
        median_size=config["MEDIAN_SIZE"],
        gaussian_sigma=config["GAUSSIAN_SIGMA"],
        background_sigma=0,
        use_log=config["DIRECT_BEAM_USE_LOG"]
    )

    beam_positions, beam_scores, beam_template = track_direct_beam_stack(
        beam_tracking_stack=beam_tracking_stack,
        manual_center_yx=config["MANUAL_CENTER_YX"],
        template_half_size=config["DIRECT_BEAM_TEMPLATE_HALF_SIZE"],
        search_radius=config["DIRECT_BEAM_SEARCH_RADIUS"],
        subpixel_neighborhood=config["SUBPIXEL_NEIGHBORHOOD"],
        ref_frame_index=config["REFERENCE_FRAME_INDEX"]
    )

    centered_stack, centering_shifts, image_center_yx = center_stack_by_direct_beam(
        stack, beam_positions
    )

    processed_stack = preprocess_stack(
        centered_stack,
        median_size=config["MEDIAN_SIZE"],
        gaussian_sigma=config["GAUSSIAN_SIGMA"],
        background_sigma=config["BACKGROUND_SIGMA"],
        use_log=config["USE_LOG_SCALE_FOR_DETECTION"]
    )

    ref_proc = processed_stack[config["REFERENCE_FRAME_INDEX"]]

    ref_spots = detect_reference_spots(
        ref_proc,
        num_spots=config["NUM_SPOTS"],
        min_distance=config["MIN_DISTANCE_BETWEEN_SPOTS"],
        threshold_rel=config["PEAK_THRESHOLD_REL"],
        exclude_center_radius=config["EXCLUDE_CENTER_RADIUS"],
        exclude_edge_margin=config["EXCLUDE_EDGE_MARGIN"],
        center=image_center_yx
    )

    ref_spots, templates = build_spot_templates(
        ref_proc,
        ref_spots,
        template_half_size=config["TEMPLATE_HALF_SIZE"]
    )

    tracked, scores = track_bragg_spots_stack(
        processed_stack=processed_stack,
        ref_spots=ref_spots,
        templates=templates,
        search_radius=config["SEARCH_RADIUS"],
        subpixel_neighborhood=config["SUBPIXEL_NEIGHBORHOOD"],
        ref_frame_index=config["REFERENCE_FRAME_INDEX"]
    )

    frame_centers = compute_frame_centers(tracked)

    df = build_tracking_dataframe(
        tracked=tracked,
        scores=scores,
        ref_spots=ref_spots,
        frame_centers=frame_centers,
        beam_positions=beam_positions,
        beam_scores=beam_scores,
        centering_shifts=centering_shifts,
        center_yx=image_center_yx
    )

    csv_path = os.path.join(config["OUTPUT_DIR"], "tracked_spots.csv")
    df.to_csv(csv_path, index=False)

    plot_trajectories(
        tracked,
        frame_centers,
        os.path.join(config["OUTPUT_DIR"], "trajectory_color.png"),
        mode='color'
    )

    plot_trajectories(
        tracked,
        frame_centers,
        os.path.join(config["OUTPUT_DIR"], "trajectory_minimal.png"),
        mode='minimal'
    )

    return {
        "stack": stack,
        "centered_stack": centered_stack,
        "processed_stack": processed_stack,
        "beam_positions": beam_positions,
        "beam_scores": beam_scores,
        "centering_shifts": centering_shifts,
        "ref_spots": ref_spots,
        "tracked": tracked,
        "scores": scores,
        "frame_centers": frame_centers,
        "df": df,
        "image_center_yx": image_center_yx,
    }

In [91]:
if __name__ == "__main__":
    results = run_pipeline(CONFIG)